# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library. 

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant JSON-LD schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")


## 2. Data Overview
Review the available record sets, including their `@id`s, field `@id`s, and attached data structure. This preliminary overview guides further data extraction.


In [ ]:
# List all Record Sets with their @id and provided metadata
from pprint import pprint

record_sets = [rs for rs in metadata.record_sets]
print(f"Found {len(record_sets)} record sets.")

for record_set in record_sets:
    print(f"\nRecordSet @id: {record_set.id}")
    print(f"  name: {getattr(record_set, 'name', 'N/A')}")
    print(f"  description: {getattr(record_set, 'description', 'N/A')}")
    if hasattr(record_set, 'fields'):
        print("  Fields and their @id:")
        for field in record_set.fields:
            print(f"    - {field.id}: {getattr(field, 'name', 'N/A')} (type: {getattr(field, 'data_type', 'N/A')})")
    if hasattr(record_set, 'columns'):
        print("  Columns and their @id:")
        for column in record_set.columns:
            print(f"    - {column.id}: {getattr(column, 'name', 'N/A')}")
    print('')


## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Record set and field `@id`s identified above are used for precise extraction.

In [ ]:
# Extract data from the main protein records set (replace below with the actual main RecordSet `@id`)
# If there are multiple record sets, they will appear in the cell above. Here, we extract all of them into dataframes.

# For FAIR^2, the main table is usually in the first record set; collect all @ids.
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = dict()

for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records from RecordSet @id: {rs_id}")
    except Exception as e:
        print(f"Failed to load records for {rs_id}: {e}")
        dataframes[rs_id] = pd.DataFrame()

# Let's examine columns of the first non-empty record set
main_rs_id = None
for rs_id in record_set_ids:
    if not dataframes[rs_id].empty:
        main_rs_id = rs_id
        break

if main_rs_id is not None:
    print(f"Columns in main RecordSet (@id={main_rs_id}):")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())
else:
    print("No dataframes could be loaded from RecordSets.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
Operations include removing outliers, transforming data distributions, and grouping records to prepare for further statistical and scientific analysis.


In [ ]:
# Example EDA: Filter proteins based on abundance or peptide count

# You may need to adjust the following field IDs based on exploration (see section 2 & 3 outputs)

if main_rs_id is not None:
    df = dataframes[main_rs_id]

    # Common numeric field candidates: 'peptide_count', 'abundance', 'mw', or variants—find one by previewing columns
    # We'll select the first numeric-looking column that isn't an accession/ID
    for candidate in df.columns:
        if df[candidate].dtype.kind in 'fi' and candidate.lower() != 'accession':
            numeric_field = candidate
            break
    else:
        numeric_field = df.columns[0]   # Just pick the first column if none found

    print(f"Using numeric field '{numeric_field}' for EDA.")

    threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalize the chosen numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized '{numeric_field}' for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by protein description or another categorical field if present
    group_field = None
    for col in df.columns:
        if df[col].dtype == 'object' and col.lower() != 'accession' and df[col].nunique() < 20:
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by '{group_field}':")
        display(grouped_df.head())
    else:
        print("No suitable group field found for grouping.")
else:
    print("No data loaded for EDA.")


## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here, we plot the distribution of the selected numeric field and correlations with other attributes if available.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_rs_id is not None and not filtered_df.empty:
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[numeric_field], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field} (filtered)")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # If two or more numeric fields, plot a pairwise relationship
    numeric_cols = filtered_df.select_dtypes(include=['float', 'int']).columns.tolist()
    if len(numeric_cols) >= 2:
        plt.figure(figsize=(7, 7))
        sns.scatterplot(data=filtered_df, x=numeric_cols[0], y=numeric_cols[1])
        plt.title(f"Scatter plot of {numeric_cols[0]} vs {numeric_cols[1]}")
        plt.show()
else:
    print("Not enough data to plot.")

## 6. Conclusion
This notebook showcased how to load the FAIR^2 dataset's Croissant schema, enumerate its structure, extract records according to `@id`s, and perform initial filtering, normalization, and visualization.

**Key findings:**
- The dataset supports standard protein expression analytics with named and numeric fields accessible via Croissant `@id` references.
- Filtering and normalization are straightforward with `mlcroissant` and pandas integration.
- The dataset is now ready for deeper bioinformatics or machine learning analysis.
